In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve


In [ ]:
df = pd.read_csv("S05-hw-dataset.csv")
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df["default"].value_counts()
df["default"].value_counts(normalize=True)

### ????????? ?????? ??????

??????? ???????? **3000 ??????????** ? **17 ?????????**, ??????? ??????? ?????????? `default`.
??? ???????? ???????? ????????? (`int64` ? `float64`), ??????????? ???????? ? ?????? ???.

??????? ?????????? `default` ???????? ????????.
????????????? ??????? ?????????:
- ????? `0` ? ????? **59%** ??????????;
- ????? `1` ? ????? **41%** ??????????.

????? ???????, ?????? ???????? ????????????????, ??? ?????? ??????? **ROC-AUC**
????? ?????????????, ??? accuracy, ??? ?????? ???????? ???????.


In [ ]:
y = df["default"]
X = df.drop(columns=["default", "client_id"])

In [ ]:
X.dtypes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

### ?????????? ?????? ?? ????????? ? ???????? ???????

?????? ???? ????????? ?? ????????? ? ???????? ??????? ? ????????? **80/20**.
??? ????????????????? ??????????? ??????????? ???????? `random_state=42`.

???????? `stratify=y` ???????????? ?????????? ????????? ??????? ??????? ??????????
`default` ? ????? ????????, ??? ???????? ????? ??? ??????? ?????????? ???????.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

acc_dummy = accuracy_score(y_test, y_pred_dummy)
roc_dummy = roc_auc_score(y_test, y_proba_dummy)

acc_dummy, roc_dummy

### ???????? ??????

? ???????? ??????? ?????? ??????????? `DummyClassifier` ?? ??????????
`most_frequent`, ??????? ?????? ????????????? ???????? ?????? ?????.

?????? ???????? accuracy ????? **0.59**, ??? ????????????? ????
???????? ??????? ?????? ? ???????.
???????? ROC-AUC ????? **0.5**, ??? ????????????? ?????????? ??????????.

?????? ????????? ???????????? ??? ??????? ??????? ????????,
? ??????? ????? ???????????? ????????????? ?????????.


In [ ]:
C_list = [0.01, 0.1, 1, 10, 100]
results = []
best = None

for C in C_list:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000, C=C))
    ])
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)

    results.append({"C": C, "accuracy": acc, "roc_auc": auc})

    if best is None or auc > best["roc_auc"]:
        best = {"C": C, "accuracy": acc, "roc_auc": auc, "model": model}

pd.DataFrame(results)

### ?????? ??????? ????????? C

??? ????????????? ????????? ??? ???????? ??????? ????????? ????????????? `C`.
?? ???? ????????????? ??????? ?????? ?????????? ?????????? ????????
ROC-AUC ?? ?????? ~0.875?0.876.

?????? ???????? ROC-AUC ??????????? ??? `C = 100`, ?????? ???????? ?????
?????????? `C` ??????????, ??? ??????? ? ???????????? ?????? ? ??????????
???????? ????????????.

?? ????????? ? ???????? ??????? (ROC-AUC = 0.5) ????????????? ?????????
??????????? ???????? ???????? ????????????.


In [ ]:
proba_best = best["model"].predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba_best)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={best['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.grid(True)
plt.show()

### ROC-??????

ROC-?????? ??? ?????? ?????? ????????????? ????????? ?????????????
??????? ??????????? ?????? ????????? ??????.

???????? ROC-AUC ?????????? ???????? **0.876**, ??? ??????????? ????
?????? ?????????? ?????????????? (0.5) ? ???????????? ??????? ????????
??????.


In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={best['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.grid(True)

plt.savefig("figures/roc_curve.png", dpi=150, bbox_inches="tight")
plt.close()


### ??????? ????????? Dummy vs LogReg

In [ ]:
summary = pd.DataFrame([
    {"model": "Dummy(most_frequent)", "accuracy": acc_dummy, "roc_auc": roc_dummy},
    {"model": f"LogReg(C={best['C']})", "accuracy": accuracy_score(y_test, (proba_best >= 0.5).astype(int)), "roc_auc": roc_auc_score(y_test, proba_best)},
])
summary


### ??????

? ???? ?????? ???? ?????? ?????? ???????? ????????????? ??? ????????????
??????? ?????????? `default` ?? ?????? ?????????? ?????????.
???????? ?????? `DummyClassifier` ???????? accuracy ????? **0.59** ? ROC-AUC = **0.5**,
??? ????????????? ???????????? ???????? ??????? ?????? ? ?????????? ???????? ?? ROC-AUC.
????????????? ????????? ? ?????????????? ?????????????? ?????????
??????????? ???????? ???????? ??????: ROC-AUC ????? ?? ???????? **0.876**, ? accuracy ? ?? **0.80**.
??????? ????????? ????????????? `C` ??????? ???????????? ???????? ??????,
??? ???? ????????? ???????? ROC-AUC ??????????? ??? `C = 100`, ???? ???????? ????? ?????????? `C` ??????????.
??????????? ROC-?????? ???????????? ??????? ??????????? ?????? ????????? ??????.
